In [40]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

In [41]:
csv_path = r"E:\PROJECTWORSHOP\Eggplant Leaf Disease Detection Dataset\csv_data\glcm_features_all_diseases.csv"

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print(df.head())


Shape: (4089, 17)
           Name  contrast_0  contrast_45  contrast_90  contrast_135  \
0  healthy_leaf   84.132378     0.977386     0.525272      0.589395   
1  healthy_leaf   57.793619     0.993009     0.700724      0.746400   
2  healthy_leaf   54.647851     0.994330     0.747336      0.788399   
3  healthy_leaf   76.295438     0.992131     0.701964      0.739887   
4  healthy_leaf   58.285852     0.996264     0.636349      0.700618   

   correlation_0  correlation_45  correlation_90  correlation_135   energy_0  \
0      67.180230        0.981944        0.525277         0.594382  74.770819   
1      47.542437        0.994251        0.700284         0.747166  60.951805   
2      61.645397        0.993606        0.746614         0.785743  59.862028   
3      68.485924        0.992938        0.701312         0.739131  75.423485   
4      52.331056        0.996647        0.635550         0.697256  48.555765   

   energy_45  energy_90  energy_135  homogeneity_0  homogeneity_45  \
0   

In [42]:
y = df.iloc[:, 0]
X = df.iloc[:, 1:]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

n_classes = len(np.unique(y_encoded))
print("Classes:", label_encoder.classes_)

Classes: ['healthy_leaf' 'insect_pest_disease' 'leaf_spot_disease'
 'mosaic_virus_disease' 'white_mold_disease' 'wilt_disease']


In [43]:
models = {
    "SVM": SVC(kernel="rbf", C=1.0, gamma="scale"),
    
    "LogisticRegression": LogisticRegression(
        max_iter=5000,
        n_jobs=-1,
        random_state=42
    ),
    
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softmax",
        num_class=n_classes,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ),
    
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="MultiClass",
        verbose=0,
        random_state=42
    ),
    
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.1,
        num_leaves=31,
        objective="multiclass",
        random_state=42,
        n_jobs=-1
    )
}


In [44]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
confusion_results = {}

In [45]:
for model_name, model in models.items():
    print(f"\n================ {model_name} ================")

    acc_list, prec_list, rec_list, f1_list = [], [], [], []
    cms = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_encoded), 1):
        X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc_list.append(accuracy_score(y_test, y_pred))
        prec_list.append(precision_score(y_test, y_pred, average="macro", zero_division=0))
        rec_list.append(recall_score(y_test, y_pred, average="macro", zero_division=0))
        f1_list.append(f1_score(y_test, y_pred, average="macro", zero_division=0))

        cms.append(confusion_matrix(y_test, y_pred))

    results[model_name] = {
        "Accuracy": np.mean(acc_list),
        "Precision": np.mean(prec_list),
        "Recall": np.mean(rec_list),
        "F1": np.mean(f1_list)
    }

    confusion_results[model_name] = np.mean(cms, axis=0)

    print(f"Accuracy : {results[model_name]['Accuracy']:.4f}")
    print(f"Precision: {results[model_name]['Precision']:.4f}")
    print(f"Recall   : {results[model_name]['Recall']:.4f}")
    print(f"F1-score : {results[model_name]['F1']:.4f}")
    print("Mean Confusion Matrix:")
    print(np.round(confusion_results[model_name], 2))



================ SVM ================
Accuracy : 0.4055
Precision: 0.1381
Recall   : 0.1957
F1-score : 0.1601
Mean Confusion Matrix:
[[191.8   0.    0.   98.4   0.    0. ]
 [ 78.2   0.    0.   31.    0.    0. ]
 [ 84.8   0.    0.   35.6   0.    0. ]
 [132.6   0.    0.  139.8   0.    0. ]
 [  7.6   0.    0.    5.    0.    0. ]
 [ 12.2   0.    0.    0.8   0.    0. ]]

================ LogisticRegression ================


c:\Users\VU\anaconda3\envs\ml\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\VU\anaconda3\envs\ml\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\VU\anaconda3\envs\ml\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\VU\anaconda3\envs\ml\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave i

Accuracy : 0.3925
Precision: 0.2401
Recall   : 0.2012
F1-score : 0.1759
Mean Confusion Matrix:
[[198.8   0.    0.   91.2   0.    0.2]
 [ 79.8   0.    0.2  29.2   0.    0. ]
 [ 90.8   0.    0.   29.4   0.    0.2]
 [150.    0.2   0.2 121.2   0.    0.8]
 [  7.2   0.    0.    5.4   0.    0. ]
 [ 11.    0.2   0.    0.8   0.    1. ]]

================ RandomForest ================
Accuracy : 0.4135
Precision: 0.3504
Recall   : 0.2764
F1-score : 0.2899
Mean Confusion Matrix:
[[149.   15.8  21.4 103.    0.    1. ]
 [ 57.4  10.    6.6  34.8   0.4   0. ]
 [ 50.2   5.8  31.6  31.    0.2   1.6]
 [109.2  10.    8.2 144.2   0.    0.8]
 [  7.4   0.8   1.    3.4   0.    0. ]
 [  4.8   0.6   2.8   1.4   0.    3.4]]

================ ExtraTrees ================
Accuracy : 0.4057
Precision: 0.3308
Recall   : 0.2819
F1-score : 0.2915
Mean Confusion Matrix:
[[139.8  18.4  23.4 107.6   0.    1. ]
 [ 52.8  11.2   7.8  36.4   0.4   0.6]
 [ 50.    7.8  32.4  28.2   0.    2. ]
 [104.4  12.4   9.8 144.4   0.6   

c:\Users\VU\anaconda3\envs\ml\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000513 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 3271, number of used features: 16
[LightGBM] [Info] Start training from score -1.035814
[LightGBM] [Info] Start training from score -2.015209
[LightGBM] [Info] Start training from score -1.914907
[LightGBM] [Info] Start training from score -1.098918
[LightGBM] [Info] Start training from score -4.180828
[LightGBM] [Info] Start training from score -4.141607
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [W

...

c:\Users\VU\anaconda3\envs\ml\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

c:\Users\VU\anaconda3\envs\ml\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

c:\Users\VU\anaconda3\envs\ml\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [46]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="F1", ascending=False)
results_df_percent = results_df * 100
results_df_percent = results_df_percent.round(2)

desired_order = [
    "KNN",
    "SVM",
    "RandomForest",
    "LogisticRegression",
    "XGBoost",
    "CatBoost",
    "ExtraTrees",
    "LightGBM"
]

# Chỉ giữ các model có trong results
desired_order = [m for m in desired_order if m in results_df_percent.index]

results_df_final = results_df_percent.loc[desired_order]

print("\n========= FINAL COMPARISON (PERCENTAGE) =========")
print(results_df_final)

results_df_final.to_csv("model_comparison_results.csv")


========= FINAL COMPARISON (PERCENTAGE) =========
                    Accuracy  Precision  Recall     F1
SVM                    40.55      13.81   19.57  16.01
RandomForest           41.35      35.04   27.64  28.99
LogisticRegression     39.25      24.01   20.12  17.59
XGBoost                40.43      31.73   27.60  28.41
CatBoost               40.38      34.87   23.54  23.44
ExtraTrees             40.57      33.08   28.19  29.15
LightGBM               39.59      32.99   26.96  28.34
